# 02 · Label dan fitur K1, K2, K3

**Tujuan:** membentuk label BUY/HOLD/SELL satu jam ke depan dan tiga konfigurasi fitur.

**Input:** `output/01_harga.parquet`, `data/Kalender_Ekonomi_MT5_USD_2020_2026.csv`
**Output:** `output/02_fitur.parquet`

Padanan di kode penelitian: `features.py`, `macro_gate.py`, `macro_features.py`.

Aturan waktu untuk bar ke-*t*:
- bar dibuka pada `time`, **keputusan** dibuat saat bar selesai: `decision_time = time + 1 jam`
- fitur hanya memakai data sampai `decision_time`
- label baru diketahui saat bar berikutnya selesai: `target_end_time`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

OUTPUT = Path("output")
AMBANG = 0.5          # pengali sigma utama; 0.25 dan 1.0 hanya sensitivitas
WARM_UP = 50          # bar pertama tiap segmen belum dipakai

df = pd.read_parquet(OUTPUT / "01_harga.parquet")
df.head()

,time,open,high,low,close,segment_id
0,2020-01-02 06:00:00+00:00,1520.26,1520.36,1519.39,1519.39,0
1,2020-01-02 07:00:00+00:00,1519.37,1520.36,1518.95,1520.19,0
2,2020-01-02 08:00:00+00:00,1520.20,1520.91,1519.30,1520.87,0
3,2020-01-02 09:00:00+00:00,1520.94,1521.37,1519.03,1519.11,0
4,2020-01-02 10:00:00+00:00,1519.11,1520.93,1518.38,1520.75,0


## 1. Indikator teknikal

- **RSI 14 (Wilder):** 14 perubahan pertama dirata-rata biasa, sesudahnya `rata_baru = (13 × rata_lama + nilai_baru) / 14`.
- **EMA:** diawali rata-rata *p* harga pertama, lalu `EMA = α·close + (1−α)·EMA_sebelumnya`, α = 2/(p+1).
- **MACD:** EMA12 − EMA26; sinyal = EMA9 dari MACD; histogram = MACD − sinyal.

In [2]:
def rsi_wilder(close, n=14):
    hasil = np.full(len(close), np.nan)
    if len(close) <= n:
        return hasil
    beda = np.diff(close)
    naik, turun = np.maximum(beda, 0), np.maximum(-beda, 0)
    rata_naik, rata_turun = naik[:n].mean(), turun[:n].mean()

    def nilai(g, l):
        if g == 0 and l == 0:
            return 50.0
        if l == 0:
            return 100.0
        return 100 - 100 / (1 + g / l)

    hasil[n] = nilai(rata_naik, rata_turun)
    for i in range(n + 1, len(close)):
        rata_naik = ((n - 1) * rata_naik + naik[i - 1]) / n
        rata_turun = ((n - 1) * rata_turun + turun[i - 1]) / n
        hasil[i] = nilai(rata_naik, rata_turun)
    return hasil


def ema(nilai, p):
    hasil = np.full(len(nilai), np.nan)
    if len(nilai) < p:
        return hasil
    hasil[p - 1] = nilai[:p].mean()
    alpha = 2 / (p + 1)
    for i in range(p, len(nilai)):
        hasil[i] = alpha * nilai[i] + (1 - alpha) * hasil[i - 1]
    return hasil


def macd(close):
    garis = ema(close, 12) - ema(close, 26)
    sinyal = np.full(len(close), np.nan)
    ada = np.flatnonzero(np.isfinite(garis))
    sinyal[ada] = ema(garis[ada], 9)
    return garis, sinyal, garis - sinyal

## 2. Fitur per segmen

Semua fitur dihitung **di dalam tiap segmen**, jadi indikator mulai ulang setelah gap tak terjelaskan.

| Konfigurasi | Kolom |
|---|---|
| K1 (9) | log-return r(t)…r(t−5), ln(C/O), ln(H/L), σ24 |
| K2 (13) | K1 + RSI14, MACD/C, sinyal/C, histogram/C |

In [3]:
def fitur_segmen(seg):
    c = seg["close"].to_numpy()
    r = np.r_[np.nan, np.log(c[1:] / c[:-1])]          # log-return r(t) = ln(C_t / C_t-1)
    f = pd.DataFrame(index=seg.index)
    for lag in range(6):
        f[f"return_lag_{lag}"] = pd.Series(r, index=seg.index).shift(lag)
    f["log_close_open"] = np.log(seg["close"] / seg["open"])
    f["log_high_low"] = np.log(seg["high"] / seg["low"])
    f["sigma_24"] = pd.Series(r, index=seg.index).rolling(24).std(ddof=1)   # STDEV.S
    f["rsi_14"] = rsi_wilder(c)
    garis, sinyal, hist = macd(c)
    f["macd_line_norm"] = garis / c
    f["macd_signal_norm"] = sinyal / c
    f["macd_hist_norm"] = hist / c
    f["bar_ke"] = np.arange(1, len(seg) + 1)               # nomor bar di dalam segmen
    return f

df = df.join(pd.concat(fitur_segmen(seg) for _, seg in df.groupby("segment_id")))

K1 = ["return_lag_0", "return_lag_1", "return_lag_2", "return_lag_3", "return_lag_4",
      "return_lag_5", "log_close_open", "log_high_low", "sigma_24"]
K2 = K1 + ["rsi_14", "macd_line_norm", "macd_signal_norm", "macd_hist_norm"]

df["fitur_layak"] = (df["bar_ke"] >= WARM_UP) & df[K2].notna().all(axis=1)
df[K2].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
return_lag_0,0.000025,0.002347,-0.058846,0.026851
return_lag_1,0.000025,0.002347,-0.058846,0.026851
return_lag_2,0.000026,0.002347,-0.058846,0.026851
return_lag_3,0.000026,0.002347,-0.058846,0.026851
return_lag_4,0.000026,0.002347,-0.058846,0.026851
return_lag_5,0.000026,0.002347,-0.058846,0.026851
log_close_open,0.000023,0.002313,-0.058846,0.026891
log_high_low,0.002986,0.002572,0.000000,0.075300
sigma_24,0.002019,0.001197,0.000486,0.016320
rsi_14,51.468228,12.750452,9.136374,93.139683


## 3. Label satu bar ke depan

q = ln(C(t+1) / C(t)), δ = 0,5 × σ24

| Kondisi | Label |
|---|---|
| q > δ | BUY |
| q < −δ | SELL |
| −δ ≤ q ≤ δ | HOLD |

Label hanya sah jika bar berikutnya dibuka **tepat** 1 jam kemudian di segmen yang sama, dan σ24 > 0. Jadi label tidak pernah melompati akhir pekan atau gap.

In [4]:
waktu_berikut = df["time"].shift(-1)
df["decision_time"] = df["time"] + pd.Timedelta(hours=1)
df["target_end_time"] = waktu_berikut + pd.Timedelta(hours=1)
df["target_return"] = np.log(df["close"].shift(-1) / df["close"])
delta = AMBANG * df["sigma_24"]

df["target_layak"] = (
    (waktu_berikut - df["time"] == pd.Timedelta(hours=1))
    & (df["segment_id"].shift(-1) == df["segment_id"])
    & (df["sigma_24"] > 0)
)
df["label"] = np.select([df["target_return"] > delta, df["target_return"] < -delta], ["BUY", "SELL"], "HOLD")
df.loc[~df["target_layak"], "label"] = None

layak = df["fitur_layak"] & df["target_layak"]
print("Bar layak (fitur dan label):", layak.sum())
df.loc[layak, "label"].value_counts()

Bar layak (fitur dan label): 36658


label
HOLD    17637
BUY      9841
SELL     9180
Name: count, dtype: int64

## 4. Fitur makro (K3)

Kalender MT5 mencatat waktu rilis. Untuk snapshot ini, waktu tersebut diperlakukan sebagai UTC karena 10 jangkar waktu rilis resmi (BLS/Fed) cocok tepat.

Aturan per keluarga kejadian:
- pakai rilis **terakhir yang sudah terjadi sebelum** `decision_time` (rilis tepat pada waktu keputusan belum boleh dipakai)
- **surprise** = actual − forecast (hanya CPI, Core CPI, NFP, pengangguran)
- umur rilis dalam jam, dipotong di 168; setelah 168 jam surprise dikosongkan
- penanda rilis dalam 1, 6, dan 24 jam terakhir

**Batasan:** nilai actual/forecast diambil retrospektif; vintage independen tidak tersedia.

In [5]:
kal = pd.read_csv(Path("data") / "Kalender_Ekonomi_MT5_USD_2020_2026.csv")
kal["event_time_utc"] = pd.to_datetime(kal["event_time_server"], format="%Y.%m.%dT%H:%M:%S", utc=True)

jangkar = [("US_NFP", "2020-01-10 13:30"), ("US_CPI_MM", "2020-01-14 13:30"), ("US_NFP", "2020-07-02 12:30"),
           ("US_CPI_MM", "2020-07-14 12:30"), ("US_NFP", "2020-11-06 13:30"), ("US_CPI_MM", "2020-11-12 13:30"),
           ("US_FOMC", "2024-01-31 19:00"), ("US_NFP", "2024-07-05 12:30"), ("US_CPI_MM", "2024-07-11 12:30"),
           ("US_FOMC", "2024-07-31 18:00")]
cocok = [((kal["event_code"] == k) & (kal["event_time_utc"] == pd.Timestamp(t, tz="UTC"))).sum() == 1 for k, t in jangkar]
print("Jangkar waktu cocok:", sum(cocok), "/ 10")
kal["event_code"].value_counts()

Jangkar waktu cocok: 10 / 10


event_code
US_FOMC            186
US_UNEMPLOYMENT     78
US_NFP              78
US_CORE_CPI_YY      78
US_CPI_YY           78
US_CPI_MM           77
US_CORE_CPI_MM      77
US_RATE             54
Name: count, dtype: int64

In [6]:
ANGKA = ["US_CPI_MM", "US_CPI_YY", "US_CORE_CPI_MM", "US_CORE_CPI_YY", "US_NFP", "US_UNEMPLOYMENT"]
SEMUA_KEJADIAN = ANGKA + ["US_RATE", "US_FOMC"]

kiri = df[["decision_time"]].reset_index()
for kode in SEMUA_KEJADIAN:
    p = f"macro_{kode.lower()}"
    ev = (kal[kal["event_code"] == kode]
          .sort_values(["event_time_utc", "value_id"])
          .drop_duplicates("event_time_utc", keep="last"))
    ev = ev.assign(surprise=pd.to_numeric(ev["actual"], errors="coerce") - pd.to_numeric(ev["forecast"], errors="coerce"))
    j = pd.merge_asof(kiri, ev[["event_time_utc", "surprise"]], left_on="decision_time",
                      right_on="event_time_utc", direction="backward", allow_exact_matches=False).set_index("index")
    umur = (j["decision_time"] - j["event_time_utc"]).dt.total_seconds() / 3600
    kedaluwarsa = umur >= 168
    if kode in ANGKA:
        df[f"{p}_surprise"] = j["surprise"].where(~kedaluwarsa)
    df[f"{p}_age_hours"] = umur.clip(upper=168)
    df[f"{p}_expired"] = kedaluwarsa.astype(int)
    for jam in (1, 6, 24):
        df[f"{p}_within_{jam}h"] = ((umur > 0) & (umur <= jam)).astype(int)

MAKRO = [c for c in df.columns if c.startswith("macro_")]
print("Kolom makro:", len(MAKRO), "→ K3 =", len(K2) + len(MAKRO), "kolom")
df.loc[layak, ["decision_time", "macro_us_cpi_mm_surprise", "macro_us_cpi_mm_age_hours", "macro_us_nfp_surprise"]].tail()

Kolom makro: 46 → K3 = 59 kolom


,decision_time,macro_us_cpi_mm_surprise,macro_us_cpi_mm_age_hours,macro_us_nfp_surprise
38793,2026-07-31 18:00:00+00:00,NaN,168.0,NaN
38794,2026-07-31 19:00:00+00:00,NaN,168.0,NaN
38795,2026-07-31 20:00:00+00:00,NaN,168.0,NaN
38796,2026-07-31 21:00:00+00:00,NaN,168.0,NaN
38797,2026-07-31 22:00:00+00:00,NaN,168.0,NaN


In [7]:
df.to_parquet(OUTPUT / "02_fitur.parquet", index=False)
print("Tersimpan:", OUTPUT / "02_fitur.parquet", df.shape)

Tersimpan: output/02_fitur.parquet (38799, 72)
